# Benchmark Report Analysis

This notebook loads `benchmark_report.json`, compares agents across the main evaluation metrics, and surfaces likely performance bottlenecks from both summary metrics and trace metadata.

It is written to work in either:
- Google Colab
- a local Jupyter environment inside this repo


## What This Notebook Answers

1. Which agent performs best overall?
2. Which tasks are hardest?
3. Are failures caused by retrieval, citation use, grounding, or synthesis?
4. Do some agents spend more steps for similar or worse scores?


In [ ]:
from pathlib import Path
import json
import math

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 120)


In [ ]:
try:
    from google.colab import files  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

candidate_paths = [
    Path("../results/benchmark_report.json"),
    Path("results/benchmark_report.json"),
    Path("benchmark_report.json"),
]

report_path = next((path for path in candidate_paths if path.exists()), None)

if report_path is None and IN_COLAB:
    print("Upload benchmark_report.json from your local machine.")
    uploaded = files.upload()
    if not uploaded:
        raise FileNotFoundError("No benchmark report uploaded.")
    uploaded_name = next(iter(uploaded.keys()))
    report_path = Path(uploaded_name)

if report_path is None:
    raise FileNotFoundError("Could not find benchmark_report.json. Put the notebook under notebooks/ or upload the file in Colab.")

report = json.loads(report_path.read_text())
print(f"Loaded report from: {report_path.resolve()}")
print(f"Records: {len(report['records'])}, summaries: {len(report['summaries'])}")


In [ ]:
summary_df = pd.DataFrame(report["summaries"]).sort_values("average_score", ascending=False).reset_index(drop=True)

record_rows = []
for record in report["records"]:
    task = record["task"]
    result = record["result"]
    evaluation = record["evaluation"]
    steps = result.get("steps", [])

    answer_step = next((step for step in reversed(steps) if step.get("kind") == "answer"), {})
    answer_meta = answer_step.get("metadata", {}) if isinstance(answer_step, dict) else {}
    stop_reasons = []
    for step in steps:
        metadata = step.get("metadata", {}) if isinstance(step, dict) else {}
        stop_reason = metadata.get("stop_reason")
        if stop_reason:
            stop_reasons.append(stop_reason)

    record_rows.append({
        "task_id": task["task_id"],
        "question": task["question"],
        "agent_name": result["agent_name"],
        "score": evaluation["score"],
        "fact_coverage": evaluation["fact_coverage"],
        "retrieved_doc_hit_rate": evaluation["retrieved_doc_hit_rate"],
        "citation_coverage": evaluation["citation_coverage"],
        "support_quality": evaluation["support_quality"],
        "num_steps": len(steps),
        "num_citations": len(result.get("citations", [])),
        "num_retrieved_docs": len(result.get("retrieved_doc_ids", [])),
        "output_contract_ok": answer_meta.get("output_contract_ok"),
        "failure_reason": answer_meta.get("failure_reason"),
        "stop_reasons": ", ".join(stop_reasons),
        "notes": " | ".join(evaluation.get("notes", [])),
    })

records_df = pd.DataFrame(record_rows)
summary_df


## Aggregate Comparison

These are the top-level metrics to compare agents:
- `average_score`
- `average_fact_coverage`
- `average_retrieved_doc_hit_rate`
- `average_citation_coverage`
- `average_support_quality`
- average steps and citations as efficiency signals


In [ ]:
display(summary_df)


In [ ]:
metric_cols = [
    "average_score",
    "average_fact_coverage",
    "average_retrieved_doc_hit_rate",
    "average_citation_coverage",
    "average_support_quality",
]

plot_df = summary_df.melt(id_vars=["agent_name"], value_vars=metric_cols, var_name="metric", value_name="value")
plt.figure(figsize=(12, 5))
sns.barplot(data=plot_df, x="metric", y="value", hue="agent_name")
plt.xticks(rotation=30, ha="right")
plt.ylim(0, 1.05)
plt.title("Aggregate Agent Performance")
plt.tight_layout()
plt.show()


## Per-Task Comparison

This helps answer which questions are hardest and whether one architecture consistently wins on a specific task type.


In [ ]:
score_pivot = records_df.pivot(index="task_id", columns="agent_name", values="score").sort_index()
display(score_pivot)

plt.figure(figsize=(8, max(3, 0.8 * len(score_pivot))))
sns.heatmap(score_pivot, annot=True, cmap="YlGnBu", vmin=0.0, vmax=1.0)
plt.title("Per-Task Score by Agent")
plt.tight_layout()
plt.show()


In [ ]:
hardest_tasks = (
    records_df.groupby(["task_id", "question"], as_index=False)["score"]
    .mean()
    .rename(columns={"score": "mean_score_across_agents"})
    .sort_values("mean_score_across_agents")
)
display(hardest_tasks.head(10))


## Bottleneck Heuristics

The heuristics below are simple but useful:
- `retrieval` bottleneck: gold docs were not retrieved well
- `citation_use` bottleneck: evidence was retrieved but not cited well
- `grounding` bottleneck: citations exist but support quality is weak
- `synthesis` bottleneck: support is decent but fact coverage is still low


In [ ]:
def infer_bottleneck(row):
    if row["retrieved_doc_hit_rate"] < 0.5:
        return "retrieval"
    if row["citation_coverage"] + 1e-9 < row["retrieved_doc_hit_rate"]:
        return "citation_use"
    if row["support_quality"] < 0.5:
        return "grounding"
    if row["fact_coverage"] < 1.0:
        return "synthesis"
    return "strong"

records_df["bottleneck"] = records_df.apply(infer_bottleneck, axis=1)
bottleneck_counts = (
    records_df.groupby(["agent_name", "bottleneck"], as_index=False)
    .size()
    .rename(columns={"size": "count"})
)
display(bottleneck_counts.sort_values(["agent_name", "count"], ascending=[True, False]))

plt.figure(figsize=(10, 4))
sns.barplot(data=bottleneck_counts, x="bottleneck", y="count", hue="agent_name")
plt.title("Bottleneck Distribution by Agent")
plt.tight_layout()
plt.show()


In [ ]:
efficiency_df = records_df[["agent_name", "task_id", "score", "num_steps", "num_citations", "num_retrieved_docs"]].copy()
display(efficiency_df.sort_values(["score", "num_steps"]))

plt.figure(figsize=(7, 5))
sns.scatterplot(data=records_df, x="num_steps", y="score", hue="agent_name", style="agent_name", s=100)
plt.title("Score vs Number of Steps")
plt.tight_layout()
plt.show()


In [ ]:
failure_view = records_df[[
    "agent_name",
    "task_id",
    "score",
    "fact_coverage",
    "retrieved_doc_hit_rate",
    "citation_coverage",
    "support_quality",
    "output_contract_ok",
    "failure_reason",
    "stop_reasons",
    "notes",
    "question",
]].sort_values(["score", "agent_name"]) 
display(failure_view.head(20))


## How To Use The Output

A good workflow is:
1. Look at the aggregate summary table and identify the strongest overall agent.
2. Look at the per-task heatmap to find tasks with high disagreement across agents.
3. Use the bottleneck table to decide what to improve next:
   - retrieval problems -> better retrieval/reranking
   - citation_use problems -> better answer prompting / citation contract
   - grounding problems -> stricter citation behavior or grounded synthesis prompt
   - synthesis problems -> stronger answer generation or clearer required-fact coverage
4. Inspect low-score rows manually using `failure_view`.
